# درس ۰۹ - الگوی طراحی فراشناخت


## راه‌اندازی

این دفترچه یادداشت الگوی طراحی متاکاگنیشن را با استفاده از چارچوب Microsoft Agent نشان می‌دهد.

**پیش‌نیازها:**
- استقرار Azure OpenAI پیکربندی شده از طریق متغیرهای محیطی
- ورود به سیستم Azure CLI انجام شده است (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## متاکاگنیشن چیست؟

متاکاگنیشن به معنای **تفکر درباره فکر کردن** است. در زمینه عامل‌های هوش مصنوعی، به این معناست که عامل‌هایی ساخته شوند که بتوانند:

- **خوداندیشی** در مورد خروجی‌ها و فرآیند استدلال خود
- **شناسایی خطاها** و بازیابی به صورت مناسب به جای شکست خاموش
- **ارزیابی** اینکه آیا پاسخ‌هایشان کامل و مفید است
- **انطباق** استراتژی خود وقتی روش اولیه کار نمی‌کند (مثلاً بازگشت به سیستم پشتیبان)

یک عامل متاکاگنیتیو فقط به سوالات پاسخ نمی‌دهد — بلکه عملکرد خود را تحت نظر دارد و به سرعت تنظیم می‌کند.


## ابزارهای اصلی و پشتیبان

یک الگوی رایج متاکاگنیشن، **استراتژی جایگزین** است. عامل ابتدا از یک ابزار اصلی استفاده می‌کند؛ اگر آن شکست خورد (مثلاً خطای 404)، عامل شکست را تشخیص داده و به صورت شفاف به ابزار پشتیبان سوئیچ می‌کند.

این مشابه سیستم‌های دنیای واقعی است که در آنها خدمات اصلی ممکن است در دسترس نباشند و عوامل باید قبل از انتخاب مسیر جایگزین، مشکل را خودشان تشخیص دهند.

در ادامه، دو ابزار جستجوی پرواز تعریف می‌کنیم:
- **اصلی** — پوشش‌دهنده‌ی پاریس، توکیو و بارسلونا
- **پشتیبان** — پوشش‌دهنده‌ی برلین، سیدنی و نیویورک سیتی


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## عامل خودبازتابی با بازیابی خطا

عامل زیر دستور گرفته است که ابتدا سیستم پرواز اصلی را امتحان کند، خطاها را شناسایی کند و به صورت شفاف به سیستم پشتیبان بازگردد. پس از هر پاسخ، به‌طور مختصر خود را ارزیابی می‌کند که آیا به طور کامل به سؤال کاربر پاسخ داده است یا خیر.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## الگوی خودارزیابی

جنبه دیگری از فراشناخت، **خودارزیابی** است: یک عامل جداگانه (یا همان عامل در دور دوم) پاسخ را از نظر جامعیت، دقت و سودمندی بازبینی می‌کند.

در ادامه یک عامل `ResponseEvaluator` می‌سازیم که پاسخ‌های عامل سفر را در سه بُعد امتیازدهی می‌کند.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## خلاصه

در این درس یاد گرفتید چگونه **عامل‌های فراشناختی** را با استفاده از Microsoft Agent Framework بسازید:

- **خودبازتابی**: عامل‌هایی که استدلال خود را نظارت می‌کنند و به صورت شفاف اعلام می‌کنند چه اتفاقی افتاده است.
- **بازیابی خطا با راه‌حل‌های جایگزین**: الگوی استفاده از ابزار اصلی و پشتیبان که در آن عامل خطاها را شناسایی می‌کند (مثلاً ارور ۴۰۴) و به صورت خودکار منبع جایگزین را امتحان می‌کند.
- **خودارزیابی**: عامل ارزیابی جدایی که پاسخ‌ها را از نظر کامل بودن، دقت و مفید بودن امتیازدهی می‌کند.

این الگوها باعث می‌شوند عامل‌ها مقاوم‌تر، شفاف‌تر و قابل اعتمادتر شوند — ویژگی‌های حیاتی برای پیاده‌سازی در محیط تولید.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
